# 8. Can two averages beat one?

**Open:** restart R; save the supplied script as `scripts/my-08-first-tree.R`. Finish Chapter 7 first.

**Today:** build a rule by hand, fit a real tree, look at its branches and make two predictions. We will choose its depth in Chapter 9; no test score is needed today.

## 8.1 Install once, load each session

A **package** adds functions to R. We use the package **rpart** to fit a tree.

If you have not installed it, type this in the **Console** once. It needs internet; this installation line is not a step to repeat on every run:

```r
install.packages("rpart")
```

Then run the script's `tree-package` block:

In [1]:
# If not installed yet, run install.packages("rpart") in the Console once.
library(rpart)

Installing puts the package on your computer. `library(rpart)` makes its functions available in the current R session. After restarting R, run `library(rpart)` again; you usually do not need to install again. A successful load may print nothing. If it says there is no package called rpart, run the installation line above and retry the load.

**Supplied setup — `tree-load`:** run this block to restore Chapter 7's split. Its file checks are supporting code; they do not train a model.

In [2]:
source("scripts/course_setup.R")
source("scripts/tree_helpers.R")
if (!file.exists("outputs/checkpoints/farm-split.rds")) stop("Run Chapter 7 first, or scripts/full-analysis.R, to create the baseline inputs.")
outer_split <- readRDS("outputs/checkpoints/farm-split.rds")
dir.create("outputs/tree", recursive = TRUE, showWarnings = FALSE)

## 8.2 Make two group averages yourself

These four records are **invented**. Their values make the arithmetic easy; they do not show a real effect of pasture time.

| Pasture hours/day | MSC CFU/mL | log10 MSC |
|---:|---:|---:|
| 0 | 1 | 0 |
| 4 | 1 | 0 |
| 12 | 100 | 2 |
| 16 | 100 | 2 |

Run `hand-data` to create this table:

In [3]:
# Invented records for hand arithmetic, not an rpart fit on four rows.
hand <- data.frame(pasture_hours = c(0, 4, 12, 16), MSC_CFU_ml = c(1, 1, 100, 100))
hand$log10_msc <- log10(hand$MSC_CFU_ml)
hand

pasture_hours,MSC_CFU_ml,log10_msc
<dbl>,<dbl>,<dbl>
0,1,0
4,1,0
12,100,2
16,100,2


**Guess:** what single log10 value would the Chapter 7 baseline give everyone? Run `hand-mean`:

In [4]:
hand_baseline <- mean(hand$log10_msc)
hand_baseline

[1] 1

The answer is `(0 + 0 + 2 + 2) / 4 = 1`, equivalent to 10 CFU/mL.

Now **we choose** a cut at 8 pasture hours for this hand exercise. It is not a biological threshold. Software will later compare candidate cuts; we are not fitting software to these four rows.

Run `hand-left`. The condition before the comma selects rows; the empty position after the comma keeps all columns:

In [5]:
left_group <- hand[hand$pasture_hours < 8, ]
left_group
left_mean <- mean(left_group$log10_msc)
left_mean

,pasture_hours,MSC_CFU_ml,log10_msc
,<dbl>,<dbl>,<dbl>
1,0,1,0
2,4,1,0


[1] 0

You should see the 0-hour and 4-hour records, then their mean log10 MSC: **0**.

Run `hand-right`. `>=` means “greater than or equal to”:

In [6]:
right_group <- hand[hand$pasture_hours >= 8, ]
right_group
right_mean <- mean(right_group$log10_msc)
right_mean

,pasture_hours,MSC_CFU_ml,log10_msc
,<dbl>,<dbl>,<dbl>
3,12,100,2
4,16,100,2


[1] 2

You should see the 12-hour and 16-hour records, then their mean: **2**.

```mermaid
flowchart TB
    accTitle: Two Group Means Make Predictions
    accDescr: In an invented example, a cut at eight pasture hours sends an input to a final group that predicts its own training mean on the log10 scale.
    pasture_split{"Pasture hours below 8?"}
    low_leaf["Leaf: mean log10 = 0"]
    high_leaf["Leaf: mean log10 = 2"]
    pasture_split -->|Yes| low_leaf
    pasture_split -->|No| high_leaf
```

A **leaf** is a final group. A new input follows the questions to a leaf and receives that group's training mean.

**Before running:** where does a new input of 6 hours go? Run `hand-new`. Read `ifelse(question, yes_value, no_value)` as “choose this value if yes, otherwise that value”:

In [7]:
new_hours <- 6
hand_new_prediction <- ifelse(new_hours < 8, left_mean, right_mean)
hand_new_prediction
10^hand_new_prediction

[1] 0

[1] 1

It goes left: predict 0 log10 MSC, equivalent to `10^0 = 1` CFU/mL. We do not need its MSC result to make this prediction.

## 8.3 A perfect training fit can still surprise us

Run `hand-errors`. The four group predictions are 0, 0, 2 and 2; `^2` squares each error:

In [8]:
hand$two_mean_prediction <- c(left_mean, left_mean, right_mean, right_mean)
one_mean_errors <- hand$log10_msc - hand_baseline
two_mean_errors <- hand$log10_msc - hand$two_mean_prediction
sum(one_mean_errors^2)
sum(two_mean_errors^2)

[1] 4

[1] 0

The two printed squared-error sums are **4** for one mean and **0** for the two means.

Now invent one more record, specifically to challenge the rule: it has **6 pasture hours**, but its lab result turns out to be **100 CFU/mL**. This is a constructed counterexample, not a real survey observation.

| For this new record | Baseline | Hand-built tree |
|---|---:|---:|
| Prediction, CFU/mL | 10 | 1 |
| Prediction, log10 MSC | 1 | 0 |
| Observed log10 MSC | 2 | 2 |
| Observed minus predicted | 1 | 2 |

Run `hand-surprise` to check the last row:

In [9]:
# An invented counterexample with a now-known outcome, not a real survey record.
surprise_observed <- log10(100)
surprise_observed - hand_baseline
surprise_observed - hand_new_prediction

[1] 1

[1] 2

**Explain:** which prediction missed by more on this record? A rule can perfectly match the records used to build it and still miss a new one. That is why we need hidden farms, not just a small training error.

**The baseline uses one training mean; a regression tree divides records into groups and uses each group's training mean.** Its software searches candidate cuts, then can repeat within groups. It does not search every possible complete tree.

**Pause point:** you have completed the hand-built rule and its counterexample. The rest of this chapter uses real data. If you stop here, save your my- script; next time restart R and run from the top to recreate its objects.

## 8.4 Meet the real model's two inputs

We offer the real tree two survey answers from the sampling visit:

| R column | Meaning | Role |
|---|---|---|
| `pasture_hours` | Current hours per day on pasture | Possible input |
| `milking_cows` | Number of milking cows that day | Possible input |
| `log10_msc` | Lab MSC result on the log10 scale | Answer to learn from in training |

We assume the survey answers can be obtained before the lab result; an actual application must check that timing. Farm labels help match records and separate farms; they are not prediction inputs. Another assay result or a summary of microbial outcomes would not be an appropriate replacement for these survey inputs.

**Supplied data preparation — `survey-source`:** run this block. It downloads the fixed original survey and matches each record by **farm and sampling round**, preserving the original training/test roles.

In [10]:
survey <- read_survey_features(fetch_course_survey())
train_survey <- attach_survey(outer_split$train, survey)
test_survey <- attach_survey(outer_split$test, survey)
stopifnot(identical(train_survey$log10_msc, outer_split$train$log10_msc))
stopifnot(identical(test_survey$log10_msc, outer_split$test$log10_msc))

Run `predictor-preview` to inspect three training records. Read across a row: these two inputs accompany this known training answer.

In [11]:
head(train_survey[c("pasture_hours", "milking_cows", "log10_msc")], 3)

,pasture_hours,milking_cows,log10_msc
,<dbl>,<dbl>,<dbl>
1,7,33,0.1760913
2,21,39,0.9294189
3,15,46,0.5440680


## 8.5 Give missing inputs an explicit rule

**Predict first:** does an unknown cow count mean zero cows? Run the supplied `predictor-audit` block:

In [12]:
predictor_audit <- do.call(rbind, lapply(list(train = train_survey, test = test_survey), function(data) {
  data.frame(records = nrow(data), survey_matched = sum(data$survey_matched),
    survey_unmatched = sum(!data$survey_matched), pasture_hours_missing = sum(is.na(data$pasture_hours)),
    milking_cows_missing = sum(is.na(data$milking_cows)))
}))
predictor_audit$role <- row.names(predictor_audit)
row.names(predictor_audit) <- NULL
predictor_audit <- predictor_audit[c("role", "records", "survey_matched", "survey_unmatched",
                                     "pasture_hours_missing", "milking_cows_missing")]
predictor_audit
stopifnot(nrow(train_survey) == 420, nrow(test_survey) == 105)
stopifnot(length(intersect(train_survey$FarmID, test_survey$FarmID)) == 0)
stopifnot(identical(predictor_audit$survey_unmatched, c(18L, 11L)))

role,records,survey_matched,survey_unmatched,pasture_hours_missing,milking_cows_missing
<chr>,<int>,<int>,<int>,<int>,<int>
train,420,402,18,18,20
test,105,94,11,11,11


There are **29 unmatched farm-round records**: 18 training and 11 test records. Two additional matched training records have unknown cow counts. We keep all 420 training and 105 test records for the same-sample comparison.

This is different from making up a missing lab answer:

| What is missing? | Our action | Why? |
|---|---|---|
| MSC answer | Do not invent an answer to train or score against | We do not know the correct result |
| Survey input | Fill a separate model-input copy using a training reference | The model needs a usable input; this is an explicit assumption |

Chapter 3 rejected silently treating an unknown count as zero. Here we deliberately state a different assumption for **prediction inputs**, preserve the recorded values and later evaluate the complete procedure. Filling an input does not recover its true measurement.

Our reference is the training **median**: sort the known values and take the middle one. The median of 30, 40 and 200 is 40. With an even count, average the two middle values.

Run `median-example` on these invented cow counts. `[is.na(model_cows)]` selects only the missing positions:

In [13]:
# Invented predictor values: keep the recorded values unchanged.
recorded_cows <- c(30, 40, NA, 200)
reference_cows <- median(recorded_cows, na.rm = TRUE)
model_cows <- recorded_cows
model_cows[is.na(model_cows)] <- reference_cows
data.frame(recorded = recorded_cows, used_by_model = model_cows)

recorded,used_by_model
<dbl>,<dbl>
30,30
40,40
NA,40
200,200


**Check:** the recorded column still contains NA. Only the model copy uses 40.

Now run `first-medians` on the real **training** records:

In [14]:
training_medians <- c(
  pasture_hours = median(train_survey$pasture_hours, na.rm = TRUE),
  milking_cows = median(train_survey$milking_cows, na.rm = TRUE)
)
training_medians

pasture_hours  milking_cows 
            9            72

The references are 9 hours and 72 cows. `[["milking_cows"]]` selects that named reference from `training_medians`. Run `first-fill` to copy the table and fill its missing inputs:

In [15]:
train_ready <- train_survey
train_ready$pasture_hours[is.na(train_ready$pasture_hours)] <- training_medians[["pasture_hours"]]
train_ready$milking_cows[is.na(train_ready$milking_cows)] <- training_medians[["milking_cows"]]

No test record helped calculate these references. Chapter 9 will apply the same rule separately inside each validation round.

## 8.6 Fit your first real tree and look at it

Use this object map to keep track of the names. These are objects in this R session; the diagram is a reading guide, not another block to run.

```mermaid
flowchart TB
    accTitle: From Survey Table To Predictions
    accDescr: Training survey inputs supply their own medians, which fill a separate training copy. Fitting that copy creates first_tree; applying it to invented first_inputs creates first_predictions.
    raw_table["train_survey: original training inputs"] -->|learn references| medians["training_medians: 9 hours and 72 cows"]
    raw_table -->|copy| ready_table["train_ready: filled training copy"]
    medians -->|fill missing inputs| ready_table
    ready_table -->|rpart| tree_object["first_tree: learned rule"]
    tree_object -->|predict| prediction_values["first_predictions: predicted numbers"]
    invented_inputs["first_inputs: two invented rows"] -->|newdata| prediction_values
```

`tree-load` also provides `fixed_tree_settings`, the preset settings used below. `library(rpart)` loads functions; it does not create our tables or settings. The fitting and prediction steps create the last two objects as you proceed.

**Fit** means use training records to learn a rule. Run `first-fit`:

In [16]:
first_settings <- fixed_tree_settings
first_settings$maxdepth <- 1
first_tree <- rpart(log10_msc ~ pasture_hours + milking_cows,
                    data = train_ready, method = "anova", control = first_settings)

Read the fitting call in pieces:

| Code | Meaning |
|---|---|
| `first_tree <- rpart(...)` | Learn a tree and store it as `first_tree` |
| `log10_msc ~ pasture_hours + milking_cows` | Predict log10 MSC using these two possible inputs |
| `data = train_ready` | Learn from the prepared training table |
| `method = "anova"` | Tell rpart this is a numeric prediction task |
| `first_settings$maxdepth <- 1` | Allow at most one question along a path |

Here `~` means “predict using”; `+` lists inputs, rather than adding hours to cows. The supplied `fixed_tree_settings` holds the other settings constant. Copy it, then set the visible depth. You do not need to write a settings helper. Optional details are in [the fixed settings note](../reference/data-audit.md#tree-settings).

The depth of 1 is a **starting demonstration**, not a decision that it is best. We have not calculated a tree test score.

**Supplied plot — `first-plot`:** run this block, then open `outputs/tree/first-tree.png` in Files. It saves the image directly; it may not appear in the Plots pane.

In [17]:
first_nodes <- extract_tree_nodes(first_tree)
png("outputs/tree/first-tree.png", width = 1200, height = 720, res = 150)
par(mar = c(0, 0, 3, 0), xpd = NA, family = "sans")
draw_teaching_tree(first_tree, first_nodes)
invisible(dev.off())
cat("Saved outputs/tree/first-tree.png\n")

Saved outputs/tree/first-tree.png


![A first depth-one tree learned from training data. Each leaf shows its training mean log10 MSC and number of training records.](../outputs/tree/first-tree.png)

The software chose a question about **milking cows**, not pasture hours. Its cut at 36.5 separates integer counts of **36 or fewer** from **37 or more**. There is no half cow. The cut was learned from these training records; it is not advice to change herd size or a milk-safety threshold.

## 8.7 Predict by eye, then use predict()

The next two inputs are **invented** and have no known MSC results. Run `first-inputs`:

In [18]:
# Invented inputs only: neither row has a known MSC result.
first_inputs <- data.frame(pasture_hours = c(6, 6), milking_cows = c(50, 30))
first_inputs

pasture_hours,milking_cows
<dbl>,<dbl>
6,50
6,30


Before running the next block, point to the leaf for 50 cows, then for 30 cows. What log10 value should each receive? Run `first-predict`:

In [19]:
first_predictions <- predict(first_tree, newdata = first_inputs)
data.frame(first_inputs, prediction_log10 = as.numeric(first_predictions))

pasture_hours,milking_cows,prediction_log10
<dbl>,<dbl>,<dbl>
6,50,0.6873386
6,30,1.0545454


`predict(first_tree, newdata = first_inputs)` applies the learned rule to the new rows. It does **not** train again. Neither row needs an MSC column. Their predictions are about 0.6873 and 1.0545 log10 MSC. Without observed outcomes, these rows cannot tell us whether prediction is accurate.

**Explain:** which operation learned the branches, and which just followed them? Does changing pasture hours alone change this particular tree's prediction?

**Supplied save — `save-tree-input`:** run this last block. It saves the original survey inputs, including their missing values, for Chapter 9 to prepare within each fitting round.

In [20]:
write.csv(predictor_audit, "outputs/tree/predictor-audit.csv", row.names = FALSE)
saveRDS(list(train = train_survey, test = test_survey, audit = predictor_audit),
        "outputs/checkpoints/tree-input.rds")
file.exists("outputs/checkpoints/tree-input.rds")

[1] TRUE

**Find:** `outputs/checkpoints/tree-input.rds`, `outputs/tree/predictor-audit.csv` and `outputs/tree/first-tree.png`.

**Next:** [how deep should the tree grow?](09-choose-tree.ipynb)